# Retail Data Wrangling and Analytics

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import lag
from datetime import datetime

## Load Data (File)

In [0]:
df = spark.table("jarvis_training.default.online_retail_ii")
display(df.limit(5))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom


In [0]:
display(df.describe())

summary,Invoice,StockCode,Description,Quantity,Price,Customer ID,Country
count,1067371,1067371,1062989,1067371,1067371,824364,1067371
mean,537608.1499316233,28350.201592689715,21848.25,9.9388984711033,4.649387727420796,15324.63850435002,null
stddev,26662.450446904833,17968.479697262945,922.9197780233488,172.70579407675186,123.55305872146296,1697.4644503793095,null
min,489434,10002,DOORMAT UNION JACK GUNS AND ROSES,-80995,-53594.36,12346.0,Australia
max,C581569,m,wrongly sold sets,80995,38970.0,18287.0,West Indies


## Total Invoice Amount Distribution


In [0]:
# Check for null and missing value
from pyspark.sql.functions import col, sum, when
print(df.where("quantity is null").count())
print(df.where("price is null").count())

0
0


In [0]:
# Compute the row item total
df = df.withColumn("row_amount", df["quantity"] * df["price"])

# Group the data by invoice number and compute the amount for each invoice
invoice_df = df.groupby("invoice").agg(sum('row_amount').alias("amount"))
display(invoice_df.limit(5))


invoice,amount
489446,996.1
489463,0.0
C489476,-12.600000000000001
489514,823.02
489519,696.7300000000002


In [0]:
# Exclude the non-positive amount which represents order cancellations
invoice_df = invoice_df.filter(invoice_df["amount"] > 0)

# Compute the mode, min, max, median, mean of all invoice amount
from pyspark.sql.functions import count, mean, min, max, expr

# Compute the mode
mode_value = invoice_df.groupBy("amount").count().orderBy("count", ascending=False).first()["amount"]
print(f"The mode is {mode_value}")

invoice_df.select( 
    mean("amount").alias("mean"), 
    min("amount").alias("min"),
    expr("percentile(amount, 0.5)").alias("median"),
    max("amount").alias("max")
).show()




The mode is 15.0
+-----------------+----+-----------------+--------+
|             mean| min|           median|     max|
+-----------------+----+-----------------+--------+
|523.3037611158235|0.19|304.3150000000001|168469.6|
+-----------------+----+-----------------+--------+



In [0]:
# Plot all of the amount distribution
display(invoice_df.select("amount").limit(10))

Databricks visualization. Run in Databricks to view.

amount
996.1
823.02
696.7300000000002
128.60000000000002
6.35
61.099999999999994
192.0
168.0
754.2
162.04000000000002


In [0]:
# Filter the first 85 quantiles of the invoice amount data
q85 = invoice_df.approxQuantile("amount", [0.85],0.01)[0]
remove_outliers = invoice_df.filter(invoice_df["amount"] <= q85)

# Compute the mode, min, max, median, mean of 85 quantiles invoice amount
# Compute the mode
mode = remove_outliers.groupBy("amount").count().orderBy("count", ascending=False).first()["amount"]
print(f"The mode is {mode}")

remove_outliers.select( 
    mean("amount").alias("mean"), 
    min("amount").alias("min"),
    expr("percentile(amount, 0.5)").alias("median"),
    max("amount").alias("max")
).show()

The mode is 15.0
+-----------------+----+-------+------+
|             mean| min| median|   max|
+-----------------+----+-------+------+
|268.9810316302422|0.19|254.445|707.06|
+-----------------+----+-------+------+



In [0]:
# Plot all of the amount distribution
display(remove_outliers.select("amount").limit(10))

Databricks visualization. Run in Databricks to view.

amount
696.7300000000002
128.60000000000002
6.35
61.099999999999994
192.0
168.0
162.04000000000002
319.5399999999999
221.18999999999988
316.21999999999997


## Monthly Placed and Canceled Orders

In [0]:
# Generate a new column with the invoice date as 'YYYYMM' format
df = df.withColumn("new_invoice_date", (year(col("invoicedate")) * 100 + month(col("invoicedate"))))

# Identify the cancelled order(invoice number start with 'c')
df = df.withColumn("is_cancelled",upper(col("invoice")).cast("string").startswith("C"))

display(df.limit(5))

Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,row_amount,new_invoice_date,is_cancelled
489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01T07:45:00.000Z,6.95,13085.0,United Kingdom,83.4,200912,false
489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,81.0,200912,false
489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01T07:45:00.000Z,6.75,13085.0,United Kingdom,81.0,200912,false
489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01T07:45:00.000Z,2.1,13085.0,United Kingdom,100.80000000000001,200912,false
489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01T07:45:00.000Z,1.25,13085.0,United Kingdom,30.0,200912,false


In [0]:
# Compute the monthly total orders
monthly_total_orders_df = df.groupBy("new_invoice_date").agg(countDistinct("invoice").alias("total_orders"))

# Compute the monthly cancelled orders
monthly_cancelled_orders_df = df.filter(df['is_cancelled']).groupBy("new_invoice_date").agg(countDistinct("invoice").alias("cancelled_orders"))

display(monthly_cancelled_orders_df.limit(5))

new_invoice_date,cancelled_orders
201007,344
201008,273
201012,326
201103,318
201101,260


In [0]:
# Merge the monthly total and cancelled orders into one table
monthly_placed_orders_df = monthly_total_orders_df.join(monthly_cancelled_orders_df, on = "new_invoice_date",how = "left")

# Compute the monthly placed orders
monthly_placed_orders_df = monthly_placed_orders_df.na.fill({"cancelled_orders": 0})

monthly_placed_orders_df = monthly_placed_orders_df.withColumn("placed_orders",col("total_orders") - 2 * col("cancelled_orders"))

display(monthly_placed_orders_df.limit(5))

new_invoice_date,total_orders,cancelled_orders,placed_orders
201004,1892,304,1284
201107,1927,270,1387
201106,2012,329,1354
201109,2327,333,1661
201010,2965,476,2013


In [0]:
# Plot the bar chart for placed and cancelled orders
bar_chart_df = monthly_placed_orders_df.withColumn(
    "invoice_month_date",
    to_date(concat(col("new_invoice_date").cast("string"), lit("01")), "yyyyMMdd")
).withColumn(
    "invoice_ym",
    date_format(col("invoice_month_date"), "yyyy-MM")
).select(
    "invoice_ym",
    "placed_orders",
    "cancelled_orders"
).orderBy("invoice_ym")

display(bar_chart_df.limit(10))

invoice_ym,placed_orders,cancelled_orders
2009-12,1528,401
2010-01,1033,300
2010-02,1489,240
2010-03,1553,407
2010-04,1284,304
2010-05,1604,407
2010-06,1502,357
2010-07,1329,344
2010-08,1331,273
2010-09,1633,371


Databricks visualization. Run in Databricks to view.

## Monthly Sales

In [0]:
# Extract the unique cancelled invoice number without 'c'
cancelled_invoice_no = df.filter(col("is_cancelled") == True).select(substring(col("invoice"), 2, 20).alias("invoice")).distinct()

# Filter out both cancellation and their corresponding original invoice numbers
placed_order_df = (
    df.filter(col("is_cancelled") == False)
    .join(cancelled_invoice_no, on = "invoice", how = "left_anti"))

# Compute the monthly sales amount
monthly_sales_df = (placed_order_df.groupBy("new_invoice_date").agg(sum("row_amount").alias("sales_amount")).orderBy("new_invoice_date"))

display(monthly_sales_df.limit(5))

new_invoice_date,sales_amount
200912,825685.7600000115
201001,652708.5019999943
201002,553339.7360000008
201003,833570.130999969
201004,627934.6319999798


In [0]:
# Plot the line graph for monthly sales amount
line_display_df = monthly_sales_df.withColumn(
    "invoice_month_date",
    to_date(concat(col("new_invoice_date").cast("string"), lit("01")), "yyyyMMdd")
).withColumn(
    "invoice_ym",
    date_format(col("invoice_month_date"), "yyyy-MM")
).select(
    "invoice_ym",
    "sales_amount"
).orderBy("invoice_ym")

display(line_display_df.limit(10))

Databricks visualization. Run in Databricks to view.

invoice_ym,sales_amount
2009-12,825685.7600000115
2010-01,652708.5019999943
2010-02,553339.7360000008
2010-03,833570.130999969
2010-04,627934.6319999798
2010-05,659858.8599999907
2010-06,752270.1399999794
2010-07,606681.149999991
2010-08,697274.9099999849
2010-09,924333.0109999602


## Monthly Sales Growth

In [0]:
# Compute the monthly sales growth
monthly_sales_df = monthly_sales_df.withColumn(
    "invoice_month_date",
    to_date(concat(col("new_invoice_date").cast("string"), lit("01")), "yyyyMMdd")
).withColumn(
    "invoice_ym",
    date_format(col("invoice_month_date"), "yyyy-MM"))

# Sort logic for window
w = Window.orderBy("invoice_ym")

monthly_sales_df = monthly_sales_df.withColumn(
    "prev_sales_amount",
    lag("sales_amount").over(w)
).withColumn(
    "sales_growth",
    (col("sales_amount") - col("prev_sales_amount")) / col("prev_sales_amount")
).drop("prev_sales_amount")

display(
    monthly_sales_df.select(
        "invoice_ym",
        "sales_amount",
        "sales_growth")
    .orderBy("invoice_ym").limit(10))    

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Databricks visualization. Run in Databricks to view.

invoice_ym,sales_amount,sales_growth
2009-12,825685.7600000115,null
2010-01,652708.5019999943,-0.2094952660925324
2010-02,553339.7360000008,-0.15224064907307475
2010-03,833570.130999969,0.5064346128938907
2010-04,627934.6319999798,-0.2466924993501198
2010-05,659858.8599999907,0.05084004986049544
2010-06,752270.1399999794,0.14004703975633523
2010-07,606681.149999991,-0.19353285775770981
2010-08,697274.9099999849,0.14932680865392853
2010-09,924333.0109999602,0.32563641362053347


## Monthly Active Users

In [0]:
# Compute the monthly active users based on the placed orders
monthly_active_users_df = placed_order_df.groupBy("new_invoice_date").agg(countDistinct("Customer ID").alias("active_users")).orderBy("new_invoice_date")

display(monthly_active_users_df.limit(5))

new_invoice_date,active_users
200912,955
201001,720
201002,774
201003,1057
201004,942


In [0]:
# Plot the bar graph for monthly active users
active_user_display_df = monthly_active_users_df.withColumn(
    "invoice_month_date",
    to_date(concat(col("new_invoice_date").cast("string"), lit("01")), "yyyyMMdd")
).withColumn(
    "invoice_ym",
    date_format(col("invoice_month_date"), "yyyy-MM")
).select(
    "invoice_ym",
    "active_users"
).orderBy("invoice_ym")

display(active_user_display_df.limit(10))

Databricks visualization. Run in Databricks to view.

invoice_ym,active_users
2009-12,955
2010-01,720
2010-02,774
2010-03,1057
2010-04,942
2010-05,966
2010-06,1041
2010-07,928
2010-08,911
2010-09,1145


## New and Existing Users

In [0]:
# Compute the first purchase date for each customer
first_purchase_df = placed_order_df.groupBy("Customer ID").agg(min("new_invoice_date").alias("first_purchase_date"))

display(first_purchase_df.limit(5))

Customer ID,first_purchase_date
14110.0,200912
15311.0,200912
14543.0,200912
13819.0,200912
12842.0,200912


In [0]:
# Compute the number of new users per month
new_user_count_df = first_purchase_df.groupBy("first_purchase_date").agg(countDistinct("Customer ID").alias("new_user_count")).withColumnRenamed(
"first_purchase_date","new_invoice_date")

display(new_user_count_df.limit(5))

new_invoice_date,new_user_count
200912,955
201001,383
201002,376
201003,443
201004,294


In [0]:
# Compute the number of existing users per month by merging first purchase date to the transactional dataset
ex_user_count_df = placed_order_df.join(first_purchase_df, on ="Customer ID")

ex_user_count_df = ex_user_count_df.filter(col("new_invoice_date") > col("first_purchase_date")).groupBy("new_invoice_date").agg(countDistinct("Customer ID").alias("ex_user_count"))

display(ex_user_count_df.limit(5))       

new_invoice_date,ex_user_count
201106,883
201103,795
201006,771
201009,902
201008,749


In [0]:
# Merge new users count dataset and existing user count dataset
new_ex_count_df = new_user_count_df.join(ex_user_count_df, on = "new_invoice_date", how = "outer").na.fill(0).orderBy("new_invoice_date")

display(new_ex_count_df.limit(5))

new_invoice_date,new_user_count,ex_user_count
200912,955,0
201001,383,337
201002,376,398
201003,443,614
201004,294,648


In [0]:
# Plot the bar chart for new and existing users
new_exist_user_display_df = new_ex_count_df.withColumn(
    "invoice_month_date",
    to_date(concat(col("new_invoice_date").cast("string"), lit("01")), "yyyyMMdd")
).withColumn(
    "invoice_ym",
    date_format(col("invoice_month_date"), "yyyy-MM")
).select(
    "invoice_ym",
    "new_user_count",
    "ex_user_count"
).orderBy("invoice_ym")

display(new_exist_user_display_df.limit(10))

Databricks visualization. Run in Databricks to view.

invoice_ym,new_user_count,ex_user_count
2009-12,955,0
2010-01,383,337
2010-02,376,398
2010-03,443,614
2010-04,294,648
2010-05,254,712
2010-06,270,771
2010-07,186,742
2010-08,162,749
2010-09,243,902


## Finding RFM

In [0]:
display(df.select(max(col("InvoiceDate"))))

max(InvoiceDate)
2011-12-09T12:50:00.000Z


In [0]:
# Set today as January 1st, 2012 for simplicity
today = datetime(2012, 1, 1)

# Create the RFM table
rfm_df = (df.groupBy("Customer ID")
    .agg(datediff(lit(today), max(col("InvoiceDate"))).alias("Recency"),countDistinct("invoice").alias("Invoice"),sum("row_amount").alias("Monetary")))

display(rfm_df.limit(5))

Customer ID,Recency,Invoice,Monetary
18087.0,121,21,14411.62
13635.0,90,6,2948.2200000000003
14341.0,68,20,4443.469999999999
16955.0,207,9,1261.16
17611.0,26,35,8205.379999999997


## RFM Segmentation

In [0]:
# Compute R / F / M scores

# windows for scoring
w_recency = Window.orderBy(col("Recency").asc())     
w_invoice = Window.orderBy(col("Invoice").asc(), col("Customer Id").asc())
w_monetary = Window.orderBy(col("Monetary").asc())  

rfm_score_df = (
    rfm_df
    .withColumn("Recency_tmp", ntile(5).over(w_recency))
    .withColumn(
        "RecencyScore",
         when(col("Recency_tmp") == 1, 5)
         .when(col("Recency_tmp") == 2, 4)
         .when(col("Recency_tmp") == 3, 3)
         .when(col("Recency_tmp") == 4, 2)
         .otherwise(1)
    )
     .withColumn("FrequencyScore", ntile(5).over(w_invoice))
      .withColumn("MonetaryScore", ntile(5).over(w_monetary))
     .withColumn("RFM_SCORE",concat(col("RecencyScore").cast("string"),col("FrequencyScore").cast("string"), col("MonetaryScore").cast("string ")))
     .drop("Recency_tmp")
)

display(rfm_score_df.limit(5))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Customer ID,Recency,Invoice,Monetary,RecencyScore,FrequencyScore,MonetaryScore,RFM_SCORE
17399.0,563,1,-25111.09,1,2,1,121
12918.0,649,3,-10953.5,1,3,1,131
15849.0,619,1,-5876.34,1,1,1,111
15760.0,653,5,-5795.87,1,4,1,141
16981.0,563,1,-4620.86,1,1,1,111


In [0]:
# Apply the segmentation map to the rfm dataframe
rfm_segment_df = (rfm_score_df.withColumn(
        "SegmentKey",
        concat(col("RecencyScore").cast("string"),col("FrequencyScore").cast("string")))
    .withColumn(
        "Segment",
        when(col("SegmentKey").rlike(r"^[4-5][4-5]$"), "Champions")
         .when(col("SegmentKey").rlike(r"^[3-5][3-5]$"), "Loyal Customers")
         .when(col("SegmentKey").rlike(r"^[4-5][1-3]$"), "Potential Loyalist")
         .when(col("SegmentKey").rlike(r"^5[1-2]$"), "New Customers")
         .when(col("SegmentKey").rlike(r"^[3-4][1-2]$"), "Promising")
         .when(col("SegmentKey").rlike(r"^[2-3][2-3]$"), "Need Attention")
         .when(col("SegmentKey").rlike(r"^[2-3][1-2]$"), "About To Sleep")
         .when(col("SegmentKey").rlike(r"^1[3-5]$"), "Can't Lose Them")
         .when(col("SegmentKey").rlike(r"^1[2-3]$"), "At Risk")
         .when(col("SegmentKey").rlike(r"^1[1-2]$"), "Lost")
         .otherwise("Others")
    )
)

display(rfm_segment_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Customer ID,Recency,Invoice,Monetary,RecencyScore,FrequencyScore,MonetaryScore,RFM_SCORE,SegmentKey,Segment
17399.0,563,1,-25111.09,1,2,1,121,12,At Risk
12918.0,649,3,-10953.5,1,3,1,131,13,Can't Lose Them
15849.0,619,1,-5876.34,1,1,1,111,11,Lost
15760.0,653,5,-5795.87,1,4,1,141,14,Can't Lose Them
16981.0,563,1,-4620.86,1,1,1,111,11,Lost
16151.0,621,1,-4217.59,1,1,1,111,11,Lost
14063.0,453,13,-3767.1999999999994,1,5,1,151,15,Can't Lose Them
18023.0,563,1,-3248.86,1,2,1,121,12,At Risk
17013.0,626,2,-3224.76,1,2,1,121,12,At Risk
15202.0,443,8,-2570.1800000000003,1,4,1,141,14,Can't Lose Them


In [0]:
segment_summary_df = (rfm_segment_df.withColumnRenamed("Invoice", "Frequency").groupBy("Segment")
.agg(
        mean("Recency").alias("Recency_mean"),
        count("Recency").alias("Recency_count"),
        mean("Frequency").alias("Frequency_mean"),
        count("Frequency").alias("Frequency_count"),
        mean("Monetary").alias("Monetary_mean"),
        count("Monetary").alias("Monetary_count")
    ).orderBy("Segment"))

display(segment_summary_df.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Segment,Recency_mean,Recency_count,Frequency_mean,Frequency_count,Monetary_mean,Monetary_count
About To Sleep,346.023102310231,303,1.0,303,317.23841584158396,303
At Risk,569.3550135501355,369,1.6775067750677506,369,387.7929810298103,369
Can't Lose Them,523.2867132867133,286,5.283216783216783,286,1177.807587412587,286
Champions,41.73071718538566,1478,25.100135317997292,1478,9799.469456021643,1478
Lost,601.3001876172608,533,1.0,533,218.51266791744837,533
Loyal Customers,101.03849238171613,1247,6.546110665597434,1247,1999.3385934242176,1247
Need Attention,339.4203821656051,628,2.640127388535032,628,735.0735350318473,628
Others,316.8715953307393,257,9.568093385214008,257,2833.666,257
Potential Loyalist,49.01312910284464,457,1.5798687089715535,457,523.903566739606,457
Promising,129.7012987012987,385,1.4597402597402598,385,489.9172519480524,385
